In [60]:
# =========================
# SECTION 2: IMPORTS + CONFIG
# =========================

import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf

# Data settings
DATA_PERIOD = "10y"
DATA_INTERVAL = "1d"

# Indicator settings
MOMENTUM_WINDOW = 20
VOLUME_WINDOW = 20
VOLATILITY_WINDOW = 20
SHORT_SMA_WINDOW = 20
LONG_SMA_WINDOW = 50

# Strategy thresholds
MIN_MOMENTUM_20D = 0.05       # +5% over 20 trading days
MIN_VOLUME_RATIO = 1.00       # today's volume >= 20-day average volume
MAX_VOLATILITY_20D = 0.03     # daily return standard deviation below 3%

# Risk/liquidity settings
MIN_AVG_DOLLAR_VOLUME = 10_000_000  # $10 million average daily dollar volume

# Disaster kill switch setting
DISASTER_DRAWDOWN_LIMIT = 0.45       # 45% daily equity drop

print("Config loaded.")

Config loaded.


In [61]:
# %%
# =========================
# SECTION 1: 2015-STYLE BASE STOCK UNIVERSE
# =========================

"""
Base universe:
This universe is intended to represent large, liquid, well-known US stocks
that an investor could reasonably have considered around 2015.

This helps reduce the hindsight bias from only choosing today's winners.
"""

STOCK_UNIVERSE = {
    # Technology / old large-cap tech
    "AAPL": "Technology",
    "MSFT": "Technology",
    "IBM": "Technology",
    "INTC": "Technology",
    "CSCO": "Technology",
    "ORCL": "Technology",
    "QCOM": "Technology",
    "TXN": "Technology",
    "HPQ": "Technology",

    # Internet / growth names already known by 2015
    "GOOGL": "Communication Services",
    "META": "Communication Services",
    "NFLX": "Communication Services",
    "AMZN": "Consumer Discretionary",
    "EBAY": "Consumer Discretionary",
    "BKNG": "Consumer Discretionary",

    # Consumer discretionary
    "HD": "Consumer Discretionary",
    "MCD": "Consumer Discretionary",
    "NKE": "Consumer Discretionary",
    "SBUX": "Consumer Discretionary",
    "LOW": "Consumer Discretionary",
    "DIS": "Communication Services",

    # Consumer staples
    "WMT": "Consumer Staples",
    "COST": "Consumer Staples",
    "PG": "Consumer Staples",
    "KO": "Consumer Staples",
    "PEP": "Consumer Staples",
    "PM": "Consumer Staples",
    "MO": "Consumer Staples",
    "CL": "Consumer Staples",

    # Financials
    "JPM": "Financials",
    "BAC": "Financials",
    "WFC": "Financials",
    "GS": "Financials",
    "MS": "Financials",
    "V": "Financials",
    "MA": "Financials",
    "AXP": "Financials",
    "BLK": "Financials",
    "C": "Financials",

    # Healthcare
    "JNJ": "Healthcare",
    "PFE": "Healthcare",
    "MRK": "Healthcare",
    "UNH": "Healthcare",
    "ABBV": "Healthcare",
    "AMGN": "Healthcare",
    "GILD": "Healthcare",
    "BMY": "Healthcare",
    "TMO": "Healthcare",
    "ABT": "Healthcare",

    # Energy
    "XOM": "Energy",
    "CVX": "Energy",
    "COP": "Energy",
    "SLB": "Energy",
    "HAL": "Energy",

    # Industrials
    "GE": "Industrials",
    "BA": "Industrials",
    "CAT": "Industrials",
    "HON": "Industrials",
    "UPS": "Industrials",
    "MMM": "Industrials",
    "LMT": "Industrials",
    "RTX": "Industrials",

    # Materials
    "LIN": "Materials",
    "APD": "Materials",
    "SHW": "Materials",
    "DD": "Materials",

    # Telecom / utilities / defensive
    "T": "Communication Services",
    "VZ": "Communication Services",
    "NEE": "Utilities",
    "DUK": "Utilities",
    "SO": "Utilities",

    # Real estate
    "AMT": "Real Estate",
    "PLD": "Real Estate",
    "SPG": "Real Estate"
}

TICKERS = list(STOCK_UNIVERSE.keys())

print(f"Number of stocks in 2015-style base universe: {len(TICKERS)}")
print(TICKERS)

Number of stocks in 2015-style base universe: 74
['AAPL', 'MSFT', 'IBM', 'INTC', 'CSCO', 'ORCL', 'QCOM', 'TXN', 'HPQ', 'GOOGL', 'META', 'NFLX', 'AMZN', 'EBAY', 'BKNG', 'HD', 'MCD', 'NKE', 'SBUX', 'LOW', 'DIS', 'WMT', 'COST', 'PG', 'KO', 'PEP', 'PM', 'MO', 'CL', 'JPM', 'BAC', 'WFC', 'GS', 'MS', 'V', 'MA', 'AXP', 'BLK', 'C', 'JNJ', 'PFE', 'MRK', 'UNH', 'ABBV', 'AMGN', 'GILD', 'BMY', 'TMO', 'ABT', 'XOM', 'CVX', 'COP', 'SLB', 'HAL', 'GE', 'BA', 'CAT', 'HON', 'UPS', 'MMM', 'LMT', 'RTX', 'LIN', 'APD', 'SHW', 'DD', 'T', 'VZ', 'NEE', 'DUK', 'SO', 'AMT', 'PLD', 'SPG']


In [62]:
# =========================
# SECTION 3: DATA COLLECTION
# =========================

def get_stock_data(ticker, period=DATA_PERIOD, interval=DATA_INTERVAL):
    """
    Downloads OHLCV stock data for a single ticker using yfinance.

    OHLCV:
    Open   = opening price
    High   = highest price during the period
    Low    = lowest price during the period
    Close  = closing price
    Volume = number of shares traded
    """

    data = yf.download(
        ticker,
        period=period,
        interval=interval,
        progress=False,
        auto_adjust=True
    )

    if data.empty:
        raise ValueError(f"No data returned for {ticker}")

    # Flatten columns if yfinance returns multi-index columns
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data = data.dropna()

    return data


def collect_universe_data(tickers):
    """
    Downloads data for all tickers in the stock universe.
    Returns a dictionary:

    {
        "AAPL": dataframe,
        "MSFT": dataframe,
        ...
    }
    """

    stock_data = {}

    for ticker in tickers:
        try:
            data = get_stock_data(ticker)
            stock_data[ticker] = data
            print(f"Downloaded data for {ticker}: {len(data)} rows")

        except Exception as e:
            print(f"Failed to download {ticker}: {e}")

    return stock_data


stock_data = collect_universe_data(TICKERS)

print(f"\nSuccessfully downloaded data for {len(stock_data)} stocks.")

Downloaded data for AAPL: 2514 rows
Downloaded data for MSFT: 2514 rows
Downloaded data for IBM: 2514 rows
Downloaded data for INTC: 2514 rows
Downloaded data for CSCO: 2514 rows
Downloaded data for ORCL: 2514 rows
Downloaded data for QCOM: 2514 rows
Downloaded data for TXN: 2514 rows
Downloaded data for HPQ: 2514 rows
Downloaded data for GOOGL: 2514 rows
Downloaded data for META: 2514 rows
Downloaded data for NFLX: 2514 rows
Downloaded data for AMZN: 2514 rows
Downloaded data for EBAY: 2514 rows
Downloaded data for BKNG: 2514 rows
Downloaded data for HD: 2514 rows
Downloaded data for MCD: 2514 rows
Downloaded data for NKE: 2514 rows
Downloaded data for SBUX: 2514 rows
Downloaded data for LOW: 2514 rows
Downloaded data for DIS: 2514 rows
Downloaded data for WMT: 2514 rows
Downloaded data for COST: 2514 rows
Downloaded data for PG: 2514 rows
Downloaded data for KO: 2514 rows
Downloaded data for PEP: 2514 rows
Downloaded data for PM: 2514 rows
Downloaded data for MO: 2514 rows
Downloaded

In [63]:
# =========================
# SECTION 4: INDICATOR CALCULATION
# =========================

def calculate_indicators(data):
    """
    Adds technical indicators to the dataframe.

    Indicators:
    - Daily returns
    - 20-day momentum
    - 20-day average volume
    - Volume ratio
    - 20-day volatility using standard deviation of returns
    - 20-day and 50-day moving averages
    - Average dollar volume
    """

    data = data.copy()

    # Daily returns
    data["Return"] = data["Close"].pct_change()

    # Momentum: percentage change over previous 20 trading days
    data["Momentum_20D"] = data["Close"].pct_change(MOMENTUM_WINDOW)

    # Volume ratio: today's volume divided by average volume
    data["Avg_Volume_20D"] = data["Volume"].rolling(VOLUME_WINDOW).mean()
    data["Volume_Ratio"] = data["Volume"] / data["Avg_Volume_20D"]

    # Volatility: rolling standard deviation of daily returns
    data["Volatility_20D"] = data["Return"].rolling(VOLATILITY_WINDOW).std()

    # Trend: moving averages
    data["SMA_20"] = data["Close"].rolling(SHORT_SMA_WINDOW).mean()
    data["SMA_50"] = data["Close"].rolling(LONG_SMA_WINDOW).mean()

    # Dollar volume: price × volume
    data["Dollar_Volume"] = data["Close"] * data["Volume"]
    data["Avg_Dollar_Volume_20D"] = data["Dollar_Volume"].rolling(VOLUME_WINDOW).mean()

    return data


indicator_data = {}

for ticker, data in stock_data.items():
    indicator_data[ticker] = calculate_indicators(data)

print("Indicators calculated.")

Indicators calculated.


In [64]:
# =========================
# SECTION 5: STRATEGY LOGIC
# =========================

def evaluate_strategy(ticker, data):
    """
    Evaluates the current strategy conditions for a stock.

    Strategy checks:
    - Momentum is strong
    - Volume is healthy
    - Volatility is acceptable
    - Trend is upward
    - Liquidity is acceptable
    """

    latest = data.dropna().iloc[-1]

    momentum_20d = latest["Momentum_20D"]
    volume_ratio = latest["Volume_Ratio"]
    volatility_20d = latest["Volatility_20D"]
    sma_20 = latest["SMA_20"]
    sma_50 = latest["SMA_50"]
    avg_dollar_volume_20d = latest["Avg_Dollar_Volume_20D"]
    close_price = latest["Close"]

    # Strategy checks
    momentum_ok = momentum_20d > MIN_MOMENTUM_20D
    volume_ok = volume_ratio > MIN_VOLUME_RATIO
    volatility_ok = volatility_20d < MAX_VOLATILITY_20D
    trend_ok = sma_20 > sma_50
    liquidity_ok = avg_dollar_volume_20d > MIN_AVG_DOLLAR_VOLUME

    return {
        "Ticker": ticker,
        "Sector": STOCK_UNIVERSE.get(ticker, "Unknown"),
        "Close": close_price,
        "Momentum_20D": momentum_20d,
        "Volume_Ratio": volume_ratio,
        "Volatility_20D": volatility_20d,
        "SMA_20": sma_20,
        "SMA_50": sma_50,
        "Avg_Dollar_Volume_20D": avg_dollar_volume_20d,

        "momentum_ok": momentum_ok,
        "volume_ok": volume_ok,
        "volatility_ok": volatility_ok,
        "trend_ok": trend_ok,
        "liquidity_ok": liquidity_ok
    }


strategy_results = []

for ticker, data in indicator_data.items():
    try:
        result = evaluate_strategy(ticker, data)
        strategy_results.append(result)

    except Exception as e:
        print(f"Could not evaluate strategy for {ticker}: {e}")

strategy_df = pd.DataFrame(strategy_results)

strategy_df.head()

,Ticker,Sector,Close,Momentum_20D,Volume_Ratio,Volatility_20D,SMA_20,SMA_50,Avg_Dollar_Volume_20D,momentum_ok,volume_ok,volatility_ok,trend_ok,liquidity_ok
0,AAPL,Technology,298.209991,0.133200,0.723711,0.015197,279.843242,265.167920,1.363379e+10,True,False,True,True,True
1,MSFT,Technology,409.429993,-0.025770,0.785065,0.017309,417.448999,398.839999,1.435651e+10,False,False,True,True,True
2,IBM,Technology,218.369995,-0.123597,0.820052,0.024004,231.217577,238.722936,1.664143e+09,False,False,True,False,True
3,INTC,Technology,115.930000,0.692409,0.752465,0.075252,95.558000,68.243400,1.525366e+10,True,False,False,True,True
4,CSCO,Technology,115.529999,0.367219,3.026638,0.032952,93.090500,84.983653,2.259257e+09,True,True,False,True,True


In [65]:
# =========================
# SECTION 6: SIGNAL GENERATION + RISK ASSESSMENT
# =========================

def calculate_technical_score(row):
    """
    Calculates a simple technical score out of 9.

    Current scoring:
    - Momentum: up to 3 points
    - Trend: up to 2 points
    - Volume: up to 2 points
    - Volatility: up to 1 point
    - Liquidity: up to 1 point
    """

    score = 0

    # Momentum score
    if row["Momentum_20D"] > 0.10:
        score += 3
    elif row["Momentum_20D"] > 0.05:
        score += 2
    elif row["Momentum_20D"] > 0:
        score += 1

    # Trend score
    if row["trend_ok"]:
        score += 2

    # Volume score
    if row["Volume_Ratio"] > 1.5:
        score += 2
    elif row["Volume_Ratio"] > 1.0:
        score += 1

    # Volatility score
    if row["volatility_ok"]:
        score += 1

    # Liquidity score
    if row["liquidity_ok"]:
        score += 1

    return score


def generate_signal(row):
    """
    Converts strategy results into a trading signal.

    BUY:
    Strong technical setup and risk filters pass.

    WATCHLIST:
    Decent setup but not strong enough.

    AVOID:
    Weak setup or failed important filters.

    SELL:
    Trend is broken.
    """

    # If trend is broken, this is not a buy candidate
    if not row["trend_ok"]:
        return "SELL / AVOID"

    # Hard avoid if liquidity is poor
    if not row["liquidity_ok"]:
        return "AVOID - LOW LIQUIDITY"

    # Hard avoid if volatility is too high
    if not row["volatility_ok"]:
        return "AVOID - HIGH VOLATILITY"

    # Strong setup
    if (
        row["momentum_ok"]
        and row["volume_ok"]
        and row["volatility_ok"]
        and row["trend_ok"]
        and row["liquidity_ok"]
    ):
        return "BUY"

    # Otherwise, maybe worth watching
    return "WATCHLIST"


def assess_stock_risk(row):
    """
    Gives a simple risk label for the stock.

    This is stock-level risk, not portfolio-level risk.
    """

    risks = []

    if not row["liquidity_ok"]:
        risks.append("Low liquidity")

    if not row["volatility_ok"]:
        risks.append("High volatility")

    if row["Volume_Ratio"] < 0.75:
        risks.append("Weak volume")

    if row["Momentum_20D"] < 0:
        risks.append("Negative momentum")

    if not row["trend_ok"]:
        risks.append("Downtrend")

    if len(risks) == 0:
        return "OK"

    return ", ".join(risks)


def check_disaster_kill_switch(starting_day_equity, current_equity):
    """
    Disaster-level portfolio risk check.

    If portfolio equity drops 45% or more from start-of-day equity,
    stop trading and notify the user.

    portfolio equity = cash + market value of positions
    """

    if starting_day_equity <= 0:
        raise ValueError("starting_day_equity must be greater than 0")

    drawdown = (starting_day_equity - current_equity) / starting_day_equity

    if drawdown >= DISASTER_DRAWDOWN_LIMIT:
        return {
            "status": "DISASTER_STOP",
            "stop_trading": True,
            "notify_user": True,
            "drawdown": drawdown
        }

    return {
        "status": "OK",
        "stop_trading": False,
        "notify_user": False,
        "drawdown": drawdown
    }


# Apply scoring, signal generation, and risk assessment
strategy_df["Technical_Score"] = strategy_df.apply(calculate_technical_score, axis=1)
strategy_df["Signal"] = strategy_df.apply(generate_signal, axis=1)
strategy_df["Risk_Notes"] = strategy_df.apply(assess_stock_risk, axis=1)

# Sort best candidates first
ranked_df = strategy_df.sort_values(
    by=["Technical_Score", "Momentum_20D"],
    ascending=[False, False]
).reset_index(drop=True)

# Make the table easier to read
display_cols = [
    "Ticker",
    "Sector",
    "Close",
    "Momentum_20D",
    "Volume_Ratio",
    "Volatility_20D",
    "Technical_Score",
    "Signal",
    "Risk_Notes"
]

ranked_output = ranked_df[display_cols].copy()

# Format percentages for readability
ranked_output["Momentum_20D"] = ranked_output["Momentum_20D"].apply(lambda x: f"{x:.2%}")
ranked_output["Volatility_20D"] = ranked_output["Volatility_20D"].apply(lambda x: f"{x:.2%}")
ranked_output["Volume_Ratio"] = ranked_output["Volume_Ratio"].apply(lambda x: f"{x:.2f}")
ranked_output["Close"] = ranked_output["Close"].apply(lambda x: f"${x:.2f}")

ranked_output

,Ticker,Sector,Close,Momentum_20D,Volume_Ratio,Volatility_20D,Technical_Score,Signal,Risk_Notes
0,CSCO,Technology,$115.53,36.72%,3.03,3.30%,8,AVOID - HIGH VOLATILITY,High volatility
1,PM,Consumer Staples,$191.86,22.80%,1.04,2.63%,8,BUY,OK
2,UNH,Healthcare,$399.09,26.13%,0.63,1.93%,7,WATCHLIST,Weak volume
3,GOOGL,Communication Services,$401.07,19.36%,0.75,2.65%,7,WATCHLIST,OK
4,CAT,Industrials,$920.22,19.32%,0.85,2.85%,7,WATCHLIST,OK
...,...,...,...,...,...,...,...,...,...
69,IBM,Technology,$218.37,-12.36%,0.82,2.40%,2,SELL / AVOID,"Negative momentum, Downtrend"
70,TMO,Healthcare,$448.21,-13.14%,0.85,2.55%,2,SELL / AVOID,"Negative momentum, Downtrend"
71,LMT,Industrials,$520.41,-14.33%,0.51,1.70%,2,SELL / AVOID,"Weak volume, Negative momentum, Downtrend"
72,BKNG,Consumer Discretionary,$154.48,-16.30%,0.75,2.54%,2,SELL / AVOID,"Weak volume, Negative momentum, Downtrend"


In [66]:
# =========================
# OPTIONAL: TEST DISASTER KILL SWITCH
# =========================

starting_day_equity = 10_000
current_equity = 5_400

kill_switch_result = check_disaster_kill_switch(
    starting_day_equity=starting_day_equity,
    current_equity=current_equity
)

kill_switch_result

{'status': 'DISASTER_STOP',
 'stop_trading': True,
 'notify_user': True,
 'drawdown': 0.46}

In [67]:
# %%
# =========================
# SECTION 8A: NEWS / GPT MODULE CONFIG
# =========================

NEWS_CANDIDATE_COUNT = 20
NEWS_SCORES_FILE = "news_scores.csv"

# Keep GPT/news as a helper, not the main decision-maker
NEWS_WEIGHT = 0.30
TECHNICAL_WEIGHT = 0.70

MAX_RISK_SCORE_ALLOWED = 0.75
MIN_CATALYST_SCORE_FOR_WATCHLIST = 0.60

# Safety switch: keep False until you have checked news_records
RUN_GPT_API = True

# Use a cheaper API model where possible.
# If this model name is unavailable on your account, replace it with one available in your OpenAI dashboard.
GPT_NEWS_MODEL = "gpt-5.4-mini"

In [68]:
# %%
# =========================
# SECTION 8B: SELECT BOTTOM 20 NEWS CANDIDATES
# =========================

def select_bottom_news_candidates(ranked_df, bottom_n=20):
    """
    Selects bottom-ranked stocks for GPT news/catalyst analysis.

    These are not automatic buys.
    The goal is to check whether weak stocks have recovery/catalyst potential.
    """

    candidates = ranked_df.tail(bottom_n).copy()
    candidates = candidates.reset_index(drop=True)

    return candidates


bottom_20_news_candidates = ranked_df.copy()

bottom_20_news_candidates[
    [
        "Ticker",
        "Sector",
        "Close",
        "Momentum_20D",
        "Volume_Ratio",
        "Volatility_20D",
        "Technical_Score",
        "Signal",
        "Risk_Notes"
    ]
]

,Ticker,Sector,Close,Momentum_20D,Volume_Ratio,Volatility_20D,Technical_Score,Signal,Risk_Notes
0,CSCO,Technology,115.529999,0.367219,3.026638,0.032952,8,AVOID - HIGH VOLATILITY,High volatility
1,PM,Consumer Staples,191.860001,0.227983,1.036919,0.026250,8,BUY,OK
2,UNH,Healthcare,399.089996,0.261346,0.625558,0.019277,7,WATCHLIST,Weak volume
3,GOOGL,Communication Services,401.070007,0.193590,0.753462,0.026460,7,WATCHLIST,OK
4,CAT,Industrials,920.219971,0.193244,0.845606,0.028481,7,WATCHLIST,OK
...,...,...,...,...,...,...,...,...,...
69,IBM,Technology,218.369995,-0.123597,0.820052,0.024004,2,SELL / AVOID,"Negative momentum, Downtrend"
70,TMO,Healthcare,448.209991,-0.131376,0.851156,0.025504,2,SELL / AVOID,"Negative momentum, Downtrend"
71,LMT,Industrials,520.409973,-0.143344,0.508314,0.017005,2,SELL / AVOID,"Weak volume, Negative momentum, Downtrend"
72,BKNG,Consumer Discretionary,154.479996,-0.162982,0.747881,0.025379,2,SELL / AVOID,"Weak volume, Negative momentum, Downtrend"


In [69]:
# %%
# =========================
# SECTION 8C: NEWS HEADLINE INPUT
# =========================

"""
For the first version, you can manually add headlines here.

Later, this can be replaced with:
- a news API
- RSS feed
- yfinance news
- another public news source

Keep headlines short to reduce token cost.
"""

MANUAL_HEADLINES = {
    # Example format:
    # "IBM": [
    #     "IBM announces new enterprise AI partnership...",
    #     "Analysts discuss IBM cloud growth outlook..."
    # ],
}


def get_recent_headlines_for_ticker(ticker):
    """
    Returns recent public headlines/snippets for a ticker.

    First version:
    - Uses MANUAL_HEADLINES dictionary.
    - If no headlines exist, returns an empty list.
    """

    return MANUAL_HEADLINES.get(ticker, [])

In [70]:
# %%
# =========================
# SECTION 8D: BUILD GPT INPUT RECORDS
# =========================

def build_gpt_news_input(candidates_df):
    """
    Converts candidate rows into compact records for GPT scoring.
    """

    records = []

    for _, row in candidates_df.iterrows():
        ticker = row["Ticker"]

        record = {
            "ticker": ticker,
            "sector": row.get("Sector", "Unknown"),
            "technical_score": int(row["Technical_Score"]),
            "momentum_20d": float(row["Momentum_20D"]),
            "volume_ratio": float(row["Volume_Ratio"]),
            "volatility_20d": float(row["Volatility_20D"]),
            "signal": row.get("Signal", "Unknown"),
            "risk_notes": row.get("Risk_Notes", ""),
            "recent_headlines": get_recent_headlines_for_ticker(ticker)[:5]
        }

        records.append(record)

    return records


news_records = build_gpt_news_input(bottom_20_news_candidates)

news_records[:3]

[{'ticker': 'CSCO',
  'sector': 'Technology',
  'technical_score': 8,
  'momentum_20d': 0.3672189204650518,
  'volume_ratio': 3.0266381358215693,
  'volatility_20d': 0.03295193599462744,
  'signal': 'AVOID - HIGH VOLATILITY',
  'risk_notes': 'High volatility',
  'recent_headlines': []},
 {'ticker': 'PM',
  'sector': 'Consumer Staples',
  'technical_score': 8,
  'momentum_20d': 0.22798255161829206,
  'volume_ratio': 1.0369189557206901,
  'volatility_20d': 0.026250021365710405,
  'signal': 'BUY',
  'risk_notes': 'OK',
  'recent_headlines': []},
 {'ticker': 'UNH',
  'sector': 'Healthcare',
  'technical_score': 7,
  'momentum_20d': 0.2613464097235718,
  'volume_ratio': 0.6255576862122685,
  'volatility_20d': 0.019276764863313093,
  'signal': 'WATCHLIST',
  'risk_notes': 'Weak volume',
  'recent_headlines': []}]

In [71]:
# %%
# =========================
# SECTION 8E: GPT STOCK ASSESSMENT RUBRIC
# =========================

GPT_STOCK_NEWS_RUBRIC = """
You are scoring stocks for a trading bot.

Rules:
- Do not recommend direct trades.
- Only use the provided metrics and headlines.
- If headlines are missing or weak, use low confidence.
- For technically weak bottom-ranked stocks, positive news should create a watchlist flag, not an automatic buy.
- Be conservative with turnaround claims.
- You may suggest replacement tickers if a company looks weak and better public alternatives exist.
- Replacement suggestions are ideas only. The trading bot must still check technical indicators before buying.

Assess each company using:
1. Public sentiment
2. Competitive position
3. Uniqueness / moat
4. Marketing / narrative strength
5. Positive developments
6. Negative developments
7. Catalyst potential
8. Overall watchlist/replacement potential

Scoring:
- public_sentiment_score: -1 to +1
- competitive_position_score: -1 to +1
- moat_score: 0 to 1
- marketing_narrative_score: 0 to 1
- positive_development_score: 0 to 1
- risk_score: 0 to 1
- catalyst_score: 0 to 1
- overall_news_score: -1 to +1
- confidence: 0 to 1

Decision labels:
- ignore
- watchlist
- replacement_candidate
- exclude

Return JSON only in this exact shape:

{
  "scores": [
    {
      "ticker": "AAPL",
      "public_sentiment_score": 0.0,
      "competitive_position_score": 0.0,
      "moat_score": 0.0,
      "marketing_narrative_score": 0.0,
      "positive_development_score": 0.0,
      "risk_score": 0.0,
      "catalyst_score": 0.0,
      "overall_news_score": 0.0,
      "confidence": 0.0,
      "decision_label": "ignore",
      "replacement_suggestions": ["MSFT", "GOOGL"],
      "reason": "Short reason under 25 words."
    }
  ]
}
"""

In [72]:
# =========================
# SECTION 8F: GEMINI API NEWS SCORER
# ========================

import os
import json
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

GEMINI_NEWS_MODEL = "gemini-2.5-flash-lite"


NEWS_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "scores": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string"},
                    "public_sentiment_score": {"type": "number"},
                    "competitive_position_score": {"type": "number"},
                    "moat_score": {"type": "number"},
                    "marketing_narrative_score": {"type": "number"},
                    "positive_development_score": {"type": "number"},
                    "risk_score": {"type": "number"},
                    "catalyst_score": {"type": "number"},
                    "overall_news_score": {"type": "number"},
                    "confidence": {"type": "number"},
                    "decision_label": {
                        "type": "string",
                        "enum": ["ignore", "watchlist", "replacement_candidate", "exclude"]
                    },
                    "replacement_suggestions": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "reason": {"type": "string"}
                },
                "required": [
                    "ticker",
                    "public_sentiment_score",
                    "competitive_position_score",
                    "moat_score",
                    "marketing_narrative_score",
                    "positive_development_score",
                    "risk_score",
                    "catalyst_score",
                    "overall_news_score",
                    "confidence",
                    "decision_label",
                    "replacement_suggestions",
                    "reason"
                ]
            }
        }
    },
    "required": ["scores"]
}


def score_news_with_gemini(news_records):
    """
    Sends candidate stock records to Gemini and receives structured news scores.
    """

    prompt = {
        "task": "Score these bottom-ranked stocks for news/catalyst/replacement potential.",
        "rubric": GPT_STOCK_NEWS_RUBRIC,
        "stocks": news_records
    }

    response = client.models.generate_content(
        model=GEMINI_NEWS_MODEL,
        contents=json.dumps(prompt),
        config=types.GenerateContentConfig(
            system_instruction=(
                "You are a conservative stock-news scoring assistant. "
                "Return valid JSON only. Do not recommend direct trades."
            ),
            response_mime_type="application/json",
            response_schema=NEWS_RESPONSE_SCHEMA,
        ),
    )

    result = json.loads(response.text)

    return result

In [73]:
# %%
# =========================
# SECTION 8G: SAVE GPT NEWS SCORES TO CSV
# =========================

from datetime import date

def append_news_scores_to_csv(gpt_result, filename=NEWS_SCORES_FILE):
    """
    Appends GPT scores to one growing CSV file.
    """

    today_str = date.today().isoformat()
    scores = gpt_result.get("scores", [])

    rows = []

    for item in scores:
        rows.append({
            "date": today_str,
            "ticker": item.get("ticker"),
            "public_sentiment_score": item.get("public_sentiment_score"),
            "competitive_position_score": item.get("competitive_position_score"),
            "moat_score": item.get("moat_score"),
            "marketing_narrative_score": item.get("marketing_narrative_score"),
            "positive_development_score": item.get("positive_development_score"),
            "risk_score": item.get("risk_score"),
            "catalyst_score": item.get("catalyst_score"),
            "overall_news_score": item.get("overall_news_score"),
            "confidence": item.get("confidence"),
            "decision_label": item.get("decision_label"),
            "replacement_suggestions": ", ".join(item.get("replacement_suggestions", [])),
            "reason": item.get("reason")
        })

    new_scores_df = pd.DataFrame(rows)

    if os.path.exists(filename):
        old_scores_df = pd.read_csv(filename)
        combined_df = pd.concat([old_scores_df, new_scores_df], ignore_index=True)
    else:
        combined_df = new_scores_df

    combined_df.to_csv(filename, index=False)

    return combined_df


def load_latest_news_scores(filename=NEWS_SCORES_FILE):
    """
    Loads the latest GPT score for each ticker.
    """

    if not os.path.exists(filename):
        return pd.DataFrame()

    news_df = pd.read_csv(filename)
    news_df["date"] = pd.to_datetime(news_df["date"])

    latest_scores = (
        news_df.sort_values("date")
        .groupby("ticker")
        .tail(1)
        .reset_index(drop=True)
    )

    return latest_scores

In [74]:
# %%
# =========================
# SECTION 8H: RUN DAILY GPT NEWS SCORING
# =========================

if RUN_GPT_API:
    # Start with a small test first to control cost
    test_records = news_records
    gpt_result = score_news_with_gemini(test_records)

    news_scores_df = append_news_scores_to_csv(
        gpt_result=gpt_result,
        filename=NEWS_SCORES_FILE
    )

    display(news_scores_df.tail(10))

else:
    print("RUN_GPT_API is False.")
    print("Check news_records first before spending API money.")
    print("Preview of records that would be sent to GPT:")
    display(pd.DataFrame(news_records).head(5))

,date,ticker,public_sentiment_score,competitive_position_score,moat_score,marketing_narrative_score,positive_development_score,risk_score,catalyst_score,overall_news_score,confidence,decision_label,replacement_suggestions,reason
161,2026-05-15,T,-0.1,0.1,0.3,0.2,0.0,0.7,0.1,-0.4,0.3,exclude,VZ,Telecommunications company in a downtrend with...
162,2026-05-15,LOW,-0.2,0.1,0.3,0.2,0.0,0.7,0.1,-0.4,0.3,exclude,HD,Home improvement retailer in a downtrend with ...
163,2026-05-15,NKE,-0.1,0.2,0.4,0.4,0.0,0.7,0.1,-0.3,0.3,exclude,SBUX,Apparel company in a downtrend with negative m...
164,2026-05-15,MCD,0.1,0.3,0.5,0.4,0.0,0.7,0.1,-0.2,0.3,exclude,SBUX,Restaurant chain in a downtrend with negative ...
165,2026-05-15,ABT,-0.2,0.1,0.3,0.2,0.0,0.7,0.1,-0.4,0.3,exclude,"UNH, JNJ",Healthcare company in a downtrend with negativ...
166,2026-05-15,IBM,-0.2,-0.1,0.3,0.2,0.0,0.7,0.1,-0.4,0.3,exclude,"MSFT, GOOGL",Tech company in a downtrend with negative mome...
167,2026-05-15,TMO,-0.2,0.1,0.3,0.2,0.0,0.7,0.1,-0.4,0.3,exclude,"UNH, JNJ",Healthcare company in a downtrend with negativ...
168,2026-05-15,LMT,-0.3,-0.1,0.3,0.2,0.0,0.7,0.1,-0.5,0.3,exclude,"BA, RTX",Defense company in a downtrend with weak volum...
169,2026-05-15,BKNG,-0.1,0.2,0.4,0.3,0.0,0.7,0.1,-0.3,0.3,exclude,"GOOGL, AMZN",Online travel company in a downtrend with weak...
170,2026-05-15,NFLX,0.0,0.1,0.3,0.4,0.0,0.7,0.1,-0.3,0.3,exclude,"AMZN, GOOGL",Streaming service in a downtrend with weak vol...


In [75]:
print(news_records[:3])
print(latest_news_scores)
import os

print(os.path.exists("news_scores.csv"))

[{'ticker': 'CSCO', 'sector': 'Technology', 'technical_score': 8, 'momentum_20d': 0.3672189204650518, 'volume_ratio': 3.0266381358215693, 'volatility_20d': 0.03295193599462744, 'signal': 'AVOID - HIGH VOLATILITY', 'risk_notes': 'High volatility', 'recent_headlines': []}, {'ticker': 'PM', 'sector': 'Consumer Staples', 'technical_score': 8, 'momentum_20d': 0.22798255161829206, 'volume_ratio': 1.0369189557206901, 'volatility_20d': 0.026250021365710405, 'signal': 'BUY', 'risk_notes': 'OK', 'recent_headlines': []}, {'ticker': 'UNH', 'sector': 'Healthcare', 'technical_score': 7, 'momentum_20d': 0.2613464097235718, 'volume_ratio': 0.6255576862122685, 'volatility_20d': 0.019276764863313093, 'signal': 'WATCHLIST', 'risk_notes': 'Weak volume', 'recent_headlines': []}]
         date ticker  public_sentiment_score  competitive_position_score  \
0  2026-05-15    XOM                     0.0                         0.0   
1  2026-05-15     VZ                     0.0                         0.0   
2  

In [76]:
# %%
# =========================
# SECTION 8I: EXTRACT GPT REPLACEMENT SUGGESTIONS
# =========================

def extract_replacement_suggestions(news_scores_df):
    """
    Extracts replacement tickers suggested by GPT.
    """

    suggestions = set()

    if news_scores_df.empty:
        return []

    latest_scores = load_latest_news_scores()

    for _, row in latest_scores.iterrows():
        raw_suggestions = row.get("replacement_suggestions", "")

        if pd.isna(raw_suggestions) or raw_suggestions == "":
            continue

        tickers = [
            ticker.strip().upper()
            for ticker in raw_suggestions.split(",")
            if ticker.strip() != ""
        ]

        for ticker in tickers:
            suggestions.add(ticker)

    return sorted(list(suggestions))


latest_news_scores = load_latest_news_scores()

replacement_watchlist = extract_replacement_suggestions(latest_news_scores)

replacement_watchlist

['AMZN', 'GOOGL']

In [77]:
# =========================
# SECTION 8K: COMBINE TECHNICAL + NEWS SCORES
# =========================

def combine_technical_and_news_scores(ranked_df, latest_news_scores):
    final_df = ranked_df.copy()

    if latest_news_scores.empty:
        final_df["overall_news_score"] = 0
        final_df["confidence"] = 0
        final_df["risk_score"] = 0
        final_df["decision_label"] = "no_news_score"
    else:
        final_df = final_df.merge(
            latest_news_scores,
            left_on="Ticker",
            right_on="ticker",
            how="left"
        )

    # Fill missing GPT/news scores
    final_df["overall_news_score"] = final_df["overall_news_score"].fillna(0)
    final_df["confidence"] = final_df["confidence"].fillna(0)
    final_df["risk_score"] = final_df["risk_score"].fillna(0)
    final_df["decision_label"] = final_df["decision_label"].fillna("no_news_score")

    # Normalize technical score from 0-9 into 0-1
    final_df["Technical_Score_Normalized"] = final_df["Technical_Score"] / 9

    # Convert GPT news score from -1 to +1 into 0 to 1
    final_df["News_Score_Normalized"] = (final_df["overall_news_score"] + 1) / 2

    # Confidence-adjusted news score
    final_df["Adjusted_News_Score"] = (
        final_df["News_Score_Normalized"] * final_df["confidence"]
    )

    # Combined score
    final_df["Combined_Score"] = (
        TECHNICAL_WEIGHT * final_df["Technical_Score_Normalized"]
        + NEWS_WEIGHT * final_df["Adjusted_News_Score"]
    )

    # Safety rule: high GPT risk blocks buys
    final_df.loc[
        final_df["risk_score"] > MAX_RISK_SCORE_ALLOWED,
        "Signal"
    ] = "AVOID - GPT NEWS RISK"

    final_df = final_df.sort_values(
        by="Combined_Score",
        ascending=False
    ).reset_index(drop=True)

    final_df["Combined_Rank"] = final_df.index + 1

    return final_df


final_ranked_df = combine_technical_and_news_scores(
    ranked_df=ranked_df,
    latest_news_scores=latest_news_scores
)

display(final_ranked_df[
    [
        "Ticker",
        "Sector",
        "Technical_Score",
        "overall_news_score",
        "confidence",
        "risk_score",
        "Combined_Score",
        "Signal",
        "decision_label"
    ]
].head(20))

,Ticker,Sector,Technical_Score,overall_news_score,confidence,risk_score,Combined_Score,Signal,decision_label
0,CSCO,Technology,8,0.0,0.1,0.0,0.637222,AVOID - HIGH VOLATILITY,ignore
1,PM,Consumer Staples,8,0.0,0.1,0.0,0.637222,BUY,ignore
2,EBAY,Consumer Discretionary,7,0.0,0.1,0.0,0.559444,WATCHLIST,ignore
3,BA,Industrials,7,0.0,0.1,0.0,0.559444,WATCHLIST,ignore
4,GS,Financials,7,0.0,0.1,0.0,0.559444,BUY,ignore
5,DD,Materials,7,0.0,0.1,0.0,0.559444,BUY,ignore
6,SBUX,Consumer Discretionary,7,0.0,0.1,0.0,0.559444,BUY,ignore
7,MO,Consumer Staples,7,0.0,0.1,0.0,0.559444,WATCHLIST,ignore
8,AAPL,Technology,7,0.0,0.1,0.0,0.559444,WATCHLIST,ignore
9,CAT,Industrials,7,0.0,0.1,0.0,0.559444,WATCHLIST,ignore


In [78]:
# %%
# =========================
# SECTION 8J: RE-RUN TECHNICAL SCAN WITH WATCHLIST
# =========================

def run_current_technical_scan(tickers):
    """
    Downloads latest data, calculates indicators, evaluates strategy,
    and returns a ranked dataframe.

    This is for live/paper testing, not backtesting.
    """

    scan_stock_data = collect_universe_data(tickers)

    scan_indicator_data = {}

    for ticker, data in scan_stock_data.items():
        scan_indicator_data[ticker] = calculate_indicators(data)

    scan_results = []

    for ticker, data in scan_indicator_data.items():
        try:
            result = evaluate_strategy(ticker, data)
            scan_results.append(result)

        except Exception as e:
            print(f"Could not evaluate strategy for {ticker}: {e}")

    scan_df = pd.DataFrame(scan_results)

    if scan_df.empty:
        return scan_df

    scan_df["Technical_Score"] = scan_df.apply(calculate_technical_score, axis=1)
    scan_df["Signal"] = scan_df.apply(generate_signal, axis=1)
    scan_df["Risk_Notes"] = scan_df.apply(assess_stock_risk, axis=1)

    scan_ranked_df = scan_df.sort_values(
        by=["Technical_Score", "Momentum_20D"],
        ascending=[False, False]
    ).reset_index(drop=True)

    scan_ranked_df["Technical_Rank"] = scan_ranked_df.index + 1

    return scan_ranked_df


RUN_EXPANDED_SCAN = False

if RUN_EXPANDED_SCAN:
    ACTIVE_TICKERS = sorted(list(set(TICKERS + replacement_watchlist)))

    expanded_ranked_df = run_current_technical_scan(ACTIVE_TICKERS)

    display(expanded_ranked_df.head(20))
else:
    print("RUN_EXPANDED_SCAN is False.")
    print("Replacement watchlist preview:")
    print(replacement_watchlist)

RUN_EXPANDED_SCAN is False.
Replacement watchlist preview:
['AMZN', 'GOOGL']


In [79]:
# %%
# =========================
# SECTION 9A: LOCAL PAPER TRADING CONFIG
# =========================

PAPER_PORTFOLIO_FILE = "paper_portfolio.csv"
PAPER_TRADE_LOG_FILE = "paper_trade_log.csv"

PAPER_INITIAL_CASH = 10_000

MAX_PAPER_POSITIONS = 5
MAX_POSITION_SIZE = 0.20          # max 20% of portfolio per stock
MIN_COMBINED_SCORE_TO_BUY = 0.50

PAPER_TRANSACTION_COST_RATE = 0.001
ALLOW_FRACTIONAL_SHARES = True

print("Paper trading config loaded.")

Paper trading config loaded.


In [80]:
# %%
# =========================
# SECTION 9B: LOAD / CREATE PAPER PORTFOLIO
# =========================

def load_or_create_paper_portfolio(filename=PAPER_PORTFOLIO_FILE):
    """
    Loads existing paper portfolio.
    If no portfolio exists, creates a fresh one.
    """

    if os.path.exists(filename):
        portfolio = pd.read_csv(filename)
        print("Loaded existing paper portfolio.")
        return portfolio

    portfolio = pd.DataFrame([
        {
            "Ticker": "CASH",
            "Shares": 0.0,
            "Avg_Entry_Price": 0.0,
            "Current_Price": 1.0,
            "Market_Value": PAPER_INITIAL_CASH,
            "Position_Type": "Cash"
        }
    ])

    portfolio.to_csv(filename, index=False)
    print("Created new paper portfolio.")

    return portfolio


paper_portfolio = load_or_create_paper_portfolio()

paper_portfolio

Created new paper portfolio.


,Ticker,Shares,Avg_Entry_Price,Current_Price,Market_Value,Position_Type
0,CASH,0.0,0.0,1.0,10000,Cash


In [81]:
# %%
# =========================
# SECTION 9C: PAPER TRADING HELPERS
# =========================

def get_latest_price(ticker):
    """
    Gets latest close price from yfinance.
    For paper trading only.
    """

    data = get_stock_data(ticker, period="5d", interval="1d")
    latest_price = data["Close"].iloc[-1]

    return float(latest_price)


def get_cash_from_portfolio(portfolio):
    cash_row = portfolio[portfolio["Ticker"] == "CASH"]

    if cash_row.empty:
        return 0.0

    return float(cash_row.iloc[0]["Market_Value"])


def get_total_paper_equity(portfolio):
    return float(portfolio["Market_Value"].sum())


def save_paper_portfolio(portfolio, filename=PAPER_PORTFOLIO_FILE):
    portfolio.to_csv(filename, index=False)


def append_paper_trade_log(trade, filename=PAPER_TRADE_LOG_FILE):
    trade_df = pd.DataFrame([trade])

    if os.path.exists(filename):
        old_log = pd.read_csv(filename)
        new_log = pd.concat([old_log, trade_df], ignore_index=True)
    else:
        new_log = trade_df

    new_log.to_csv(filename, index=False)

    return new_log

In [83]:
# %%
# =========================
# SECTION 9D: SELECT PAPER TRADING BUY CANDIDATES
# =========================

def select_paper_buy_candidates(combined_ranked_df):
    """
    Selects final paper-trading buy candidates from the combined technical + news table.
    """

    candidates = combined_ranked_df.copy()

    candidates = candidates[
        (candidates["Signal"] == "BUY")
        & (candidates["Combined_Score"] >= MIN_COMBINED_SCORE_TO_BUY)
        & (candidates["risk_score"] <= MAX_RISK_SCORE_ALLOWED)
        & (candidates["decision_label"] != "exclude")
    ].copy()

    candidates = candidates.sort_values(
        by="Combined_Score",
        ascending=False
    ).reset_index(drop=True)

    return candidates.head(MAX_PAPER_POSITIONS)


paper_buy_candidates = select_paper_buy_candidates(final_ranked_df)

paper_buy_candidates

,Ticker,Sector,Close,Momentum_20D,Volume_Ratio,Volatility_20D,SMA_20,SMA_50,Avg_Dollar_Volume_20D,momentum_ok,...,overall_news_score,confidence,decision_label,replacement_suggestions,reason,Technical_Score_Normalized,News_Score_Normalized,Adjusted_News_Score,Combined_Score,Combined_Rank
0,PM,Consumer Staples,191.860001,0.227983,1.036919,0.026250,169.348501,166.234061,8.783086e+08,True,...,0.0,0.1,ignore,NaN,No recent headlines to analyze for news or cat...,0.888889,0.5,0.05,0.637222,2
1,GS,Financials,968.960022,0.076622,1.033105,0.014978,932.086499,878.936801,1.712702e+09,True,...,0.0,0.1,ignore,NaN,No recent headlines to analyze for news or cat...,0.777778,0.5,0.05,0.559444,5
2,DD,Materials,50.599998,0.082353,1.000388,0.025481,47.700000,46.467400,1.797232e+08,True,...,0.0,0.1,ignore,NaN,No recent headlines to analyze for news or cat...,0.777778,0.5,0.05,0.559444,6
3,SBUX,Consumer Discretionary,106.400002,0.081741,1.254907,0.021468,102.829500,98.380600,8.183768e+08,True,...,0.0,0.1,ignore,NaN,No recent headlines to analyze for news or cat...,0.777778,0.5,0.05,0.559444,7


In [84]:
# %%
# =========================
# SECTION 9E: EXECUTE LOCAL PAPER BUYS
# =========================

from datetime import datetime

def execute_paper_buys(portfolio, buy_candidates):
    """
    Executes simulated paper buys using the selected candidates.
    Saves updated portfolio and trade log.
    """

    portfolio = portfolio.copy()

    cash = get_cash_from_portfolio(portfolio)
    total_equity = get_total_paper_equity(portfolio)

    existing_tickers = set(
        portfolio[portfolio["Ticker"] != "CASH"]["Ticker"].tolist()
    )

    available_slots = MAX_PAPER_POSITIONS - len(existing_tickers)

    if available_slots <= 0:
        print("Portfolio already has max positions.")
        return portfolio

    candidates = buy_candidates[
        ~buy_candidates["Ticker"].isin(existing_tickers)
    ].copy()

    candidates = candidates.head(available_slots)

    if candidates.empty:
        print("No new candidates to buy.")
        return portfolio

    target_position_value = total_equity * MAX_POSITION_SIZE

    for _, row in candidates.iterrows():
        ticker = row["Ticker"]

        if cash <= 0:
            print("No cash left.")
            break

        latest_price = get_latest_price(ticker)

        amount_to_buy = min(cash, target_position_value)

        transaction_cost = amount_to_buy * PAPER_TRANSACTION_COST_RATE
        investable_amount = amount_to_buy - transaction_cost

        if investable_amount <= 0:
            continue

        shares = investable_amount / latest_price

        market_value = shares * latest_price

        cash -= amount_to_buy

        new_position = {
            "Ticker": ticker,
            "Shares": shares,
            "Avg_Entry_Price": latest_price,
            "Current_Price": latest_price,
            "Market_Value": market_value,
            "Position_Type": "Stock"
        }

        portfolio = pd.concat(
            [portfolio, pd.DataFrame([new_position])],
            ignore_index=True
        )

        trade = {
            "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Ticker": ticker,
            "Action": "BUY",
            "Price": latest_price,
            "Shares": shares,
            "Trade_Value": amount_to_buy,
            "Transaction_Cost": transaction_cost,
            "Combined_Score": row.get("Combined_Score", None),
            "Signal": row.get("Signal", None),
            "Reason": "Local paper buy from Section 9D candidates"
        }

        append_paper_trade_log(trade)

        print(f"Paper bought {ticker}: ${amount_to_buy:.2f} at ${latest_price:.2f}")

    # Update cash row
    portfolio.loc[portfolio["Ticker"] == "CASH", "Market_Value"] = cash

    save_paper_portfolio(portfolio)

    return portfolio


paper_portfolio = execute_paper_buys(
    portfolio=paper_portfolio,
    buy_candidates=paper_buy_candidates
)

paper_portfolio

Paper bought PM: $2000.00 at $191.86
Paper bought GS: $2000.00 at $968.96
Paper bought DD: $2000.00 at $50.60
Paper bought SBUX: $2000.00 at $106.40


,Ticker,Shares,Avg_Entry_Price,Current_Price,Market_Value,Position_Type
0,CASH,0.000000,0.000000,1.000000,2000.0,Cash
1,PM,10.413843,191.860001,191.860001,1998.0,Stock
2,GS,2.062005,968.960022,968.960022,1998.0,Stock
3,DD,39.486167,50.599998,50.599998,1998.0,Stock
4,SBUX,18.778195,106.400002,106.400002,1998.0,Stock
